# Heterogeneous natural complementarity
New inference-only development study; all real model behavior is GPU-unverified.
Default Run All prepares a plan and verifies access only. No generation occurs until
EXECUTE is set and a specific stage selected. Review/push locally, then pin that commit.
Required: original raw pilot ZIP and user-authorized access to all four official models.
No old adapter checkpoints are needed. Never resume the closed controlled-specialization run.


In [ ]:
from pathlib import Path
import os, sys, subprocess, json, re
REPO_URL = "https://github.com/Soqoro/Pact.git"
GIT_REF = "PASTE_REVIEWED_FULL_COMMIT_SHA"
STAGE = "preflight"  # preflight, smoke, private, revisions, readout, report, export
MODEL = "Q"  # Q, L, M for private/revisions
EXECUTE = False  # enables only the selected inference stage, never a full sweep
RESUME = False  # restore latest compact snapshot after runtime reset
STOP_AFTER = None  # optional positive call-boundary pause within selected model stage
SCRATCH = Path("/content/pact-heterogeneity")
CHECKOUT = SCRATCH / "checkout"
DRIVE = Path("/content/drive/MyDrive/PACT")
RUN_ID = "pact-heterogeneous-complementarity-001"
RUN = SCRATCH / "runs" / RUN_ID
PERSISTENT = DRIVE / "heterogeneity" / RUN_ID
CACHE = SCRATCH / "cache"
# Upload the exact original raw pilot ZIP here once (renaming preserves checksum).
PILOT_DRIVE = DRIVE / "sources" / "qwen3-pilot-original-raw.zip"
PILOT = SCRATCH / "sources" / "qwen3-pilot-original-raw.zip"
HF_SECRET_NAME = "HF_TOKEN"  # authorized user credential, never printed


In [ ]:
from google.colab import drive
from urllib.parse import urlsplit
drive.mount("/content/drive")
assert re.fullmatch(r"[0-9a-fA-F]{40}", GIT_REF), "Use the reviewed full commit SHA."
u = urlsplit(REPO_URL)
assert u.scheme == "https" and u.hostname and not any((u.username,u.password,u.query,u.fragment))
SCRATCH.mkdir(parents=True, exist_ok=True)
def git(*args):
    result = subprocess.run(["git", *args], capture_output=True, text=True,
                            env={**os.environ, "GIT_TERMINAL_PROMPT":"0"})
    if result.returncode: raise RuntimeError("Git checkout failed; verify URL/revision and repository access.")
    return result.stdout.strip()
if not CHECKOUT.exists(): git("clone", "--filter=blob:none", REPO_URL, str(CHECKOUT))
assert not git("-C", str(CHECKOUT), "status", "--porcelain"), "Preserve local changes first."
assert git("-C", str(CHECKOUT), "remote", "get-url", "origin") == REPO_URL
git("-C", str(CHECKOUT), "fetch", "--depth", "1", "origin", GIT_REF)
git("-C", str(CHECKOUT), "checkout", "--detach", GIT_REF)
assert git("-C", str(CHECKOUT), "rev-parse", "HEAD").lower() == GIT_REF.lower()
os.chdir(CHECKOUT)
sys.path.insert(0, str(CHECKOUT / "src"))
os.environ["TOKENIZERS_PARALLELISM"] = "false"
from pact.colab import install_dependencies
install_dependencies(CHECKOUT)


In [ ]:
from pact.storage import storage_operation
from pact.util import read_json, digest, file_hash
from pact.studies.heterogeneity_plan import PILOT_SHA

def study(stage, *options):
    command = [sys.executable, "-u", "-m", "pact.studies.heterogeneity", stage,
               "--run-dir", str(RUN), "--persistent", str(PERSISTENT), *map(str, options)]
    print("Starting:", stage, flush=True)
    subprocess.run(command, check=True)  # stream progress; don't capture hidden errors

if RESUME and not RUN.exists(): study("restore")
if not (RUN / "plan.json").exists():
    assert PILOT_DRIVE.is_file(), f"Required original raw pilot ZIP missing: {PILOT_DRIVE}"
    PILOT.parent.mkdir(parents=True, exist_ok=True)
    if not PILOT.exists():
        print(storage_operation("bundle-restore", PILOT_DRIVE, PILOT,
                                sha256=PILOT_SHA, timeout_seconds=600))
    assert file_hash(PILOT) == PILOT_SHA
    study("plan", "--pilot-bundle", PILOT)
PLAN_HASH = digest(read_json(RUN / "plan.json"))
print("PLAN_HASH:", PLAN_HASH)
print("Budget:", read_json(RUN / "plan.json")["budget"])
assert read_json(RUN / "state.json")["recovery_safe"], "Unsafe snapshot: stop; do not reset the flag."


In [ ]:
# Metadata/tokenizer access checks only. Accept Llama's terms yourself on its official page first.
if STAGE not in ("report", "export"):
    from google.colab import userdata
    import getpass
    try: token = userdata.get(HF_SECRET_NAME)
    except Exception: token = getpass.getpass("Authorized Hugging Face token (hidden): ")
    if token: os.environ["HF_TOKEN"] = token
    token = None
    study("preflight", "--execute", "--plan-hash", PLAN_HASH, "--cache-dir", CACHE)


In [ ]:
# Run only the explicitly selected stage. Default preflight stops here.
COMMON = ["--execute", "--plan-hash", PLAN_HASH, "--cache-dir", CACHE]
if STAGE in ("report", "export"):
    study(STAGE)
elif STAGE != "preflight":
    assert EXECUTE, "Set EXECUTE=True only for the stage you intend to run."
    assert STAGE in ("smoke", "private", "revisions", "readout")
    options = COMMON + (["--model", MODEL] if STAGE in ("private", "revisions") else [])
    if STOP_AFTER is not None: options += ["--stop-after", STOP_AFTER]
    study(STAGE, *options)
    study("report")
else:
    print("Plan and all-model access preflight only. No benchmark calls dispatched.")


## Explicit order after preflight
Optional `smoke` first (60 planned calls, reused). Then select and run individually:
`private Q`, `private L`, `private M`, `revisions Q`, `revisions L`, `revisions M`,
`readout`, `report`, `export`. All tasks proceed regardless of measured diversity.
After a reset use the same Git revision and RESUME=True; latest unsafe snapshots stop.
Keep both final PRIVATE and SANITIZED ZIPs and their printed SHA256 values; return
both for local review. Do not publish PRIVATE. Full commands: docs/heterogeneity_colab.md.
